In [1]:
import sys, os
import mujoco
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import time
import cv2
from PIL import Image, ImageDraw
from IPython.display import display
from OpenGL.GL import *
import time
sys.path.append('../../package/kinematics_helper/')
sys.path.append('../../package/mujoco_helper/')
sys.path.append('../../package/utility/')

from ik import *
from mujoco_parser import *
from utils import *
from transforms import *
from ik_utils import *
import datetime
from ik_utils import interpolate_and_smooth_nd

print ("MuJoCo:[%s]"%(mujoco.__version__))
"""
requirements:
mujoco
imageio
matplotlib
"""

(CVXPY) Mar 09 04:30:17 PM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.12.4544). Expected < 9.12.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Mar 09 04:30:17 PM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.12.4544). Expected < 9.12.0. Please open a feature request on cvxpy to enable support for this version.')
MuJoCo:[3.3.0]


'\nrequirements:\nmujoco\nimageio\nmatplotlib\n'

# Prerequisite

In [2]:
"""codes to make glfw use GPU in rendering time"""

os.environ["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"


In [3]:
"""canvas width, canvas height"""

canvas_width=0.125
canvas_height=0.179

In [4]:
def rpy_deg2r(r):
    r_rad=np.deg2rad(r)
    return rpy2r(r_rad)

# UR5e-2F85

In [5]:
tutorial=True

In [6]:
if tutorial:
    ur5e_path = "../../asset/universal_robots_ur5e/scene_2f85.xml"

    env=MuJoCoParserClass(name='UR5e-2F85', rel_xml_path=ur5e_path,verbose=True)


-----------------------------------------------------------------------------
name:[UR5e-2F85] dt:[0.002] HZ:[500]
 n_qpos:[14] n_qvel:[14] n_qacc:[14] n_ctrl:[7]
 integrator:[IMPLICITFAST]

n_body:[24]
 [0/24] [world] mass:[0.00]kg
 [1/24] [base] mass:[4.00]kg
 [2/24] [shoulder_link] mass:[3.70]kg
 [3/24] [upper_arm_link] mass:[8.39]kg
 [4/24] [forearm_link] mass:[2.27]kg
 [5/24] [wrist_1_link] mass:[1.22]kg
 [6/24] [wrist_2_link] mass:[1.22]kg
 [7/24] [wrist_3_link] mass:[0.19]kg
 [8/24] [end_effector_target] mass:[0.00]kg
 [9/24] [attachment] mass:[0.00]kg
 [10/24] [g2f85_base_mount] mass:[0.15]kg
 [11/24] [g2f85_base] mass:[0.78]kg
 [12/24] [g2f85_right_driver] mass:[0.01]kg
 [13/24] [g2f85_right_coupler] mass:[0.01]kg
 [14/24] [g2f85_right_spring_link] mass:[0.02]kg
 [15/24] [g2f85_right_follower] mass:[0.01]kg
 [16/24] [g2f85_right_pad] mass:[0.00]kg
 [17/24] [g2f85_right_silicone_pad] mass:[0.00]kg
 [18/24] [g2f85_left_driver] mass:[0.01]kg
 [19/24] [g2f85_left_coupler] mass:[0

## Forward Kinematics / Dynamics of UR5e-2F85

In [7]:
# if tutorial:
#     env.open_interactive_viewer()

In [8]:
if tutorial:
    rev_and_pri_joint_idxs=np.where(np.isin(env.joint_types, 
                                            [mujoco.mjtJoint.mjJNT_HINGE, mujoco.mjtJoint.mjJNT_SLIDE]))[0].astype(np.int32)
    rev_and_pri_joint_names = [env.joint_names[joint_idx] for joint_idx in rev_and_pri_joint_idxs]
    rev_and_pri_joint_mins = env.joint_ranges[rev_and_pri_joint_idxs,0]
    rev_and_pri_joint_maxs = env.joint_ranges[rev_and_pri_joint_idxs,1]
    n_rev_and_pri_joints = len(rev_and_pri_joint_names)

### FK

In [9]:
if tutorial:
    # Initialize slider control
    env.reset(step=True)
    # init_qpos = env.get_qpos_joints(joint_names=env.rev_joint_names)
    init_qpos = env.get_qpos_joints(joint_names=rev_and_pri_joint_names)
    sliders = MultiSliderClass(
        n_slider      = n_rev_and_pri_joints,
        title         = 'Sliders for [%s] Control'%(env.name),
        window_width  = 600,
        window_height = 800,
        x_offset      = 50,
        y_offset      = 100,
        slider_width  = 350,
        label_texts   = rev_and_pri_joint_names,
        slider_mins   = rev_and_pri_joint_mins,
        slider_maxs   = rev_and_pri_joint_maxs,
        slider_vals   = init_qpos,
        verbose       = False,
    )
    idxs_fwd = env.get_idxs_fwd(joint_names=rev_and_pri_joint_names)
    # Loop
    env.init_viewer(transparent=True)
    while env.is_viewer_alive():
        # Update
        sliders.update()
        slider_values = sliders.get_slider_values()
        slider_values[:6] = [-0.817,-0.691,0.66,-1.63,-1.57,-3.2]
        env.forward(q=sliders.get_slider_values(),joint_idxs=idxs_fwd)

        # Render
        if env.loop_every(tick_every=10):
            env.plot_joint_axis(axis_len=0.025,axis_r=0.005) # revolute joints
            env.plot_links_between_bodies(rgba=(0,0,0,1),r=0.001) # link information
            env.plot_contact_info()
            env.render()
    # Close slider
    sliders.close()
    print ("Done.")

env:[UR5e-2F85] reset
env:[UR5e-2F85] initalize viewer
Done.


### FD

In [10]:
# if tutorial:
#     init_ctrl = env.get_ctrl(ctrl_names=env.ctrl_names)
#     sliders = MultiSliderClass(
#         n_slider      = env.n_ctrl,
#         title         = 'Sliders for [%s] Control'%(env.name),
#         window_width  = 600,
#         window_height = 350,
#         x_offset      = 50,
#         y_offset      = 100,
#         slider_width  = 300,
#         label_texts   = env.ctrl_names,
#         slider_mins   = env.ctrl_mins,
#         slider_maxs   = env.ctrl_maxs,
#         slider_vals   = init_ctrl,
#         verbose       = False,
#     )
#     # Arrange objects
#     env.reset(step=True)
#     obj_names = env.get_body_names(prefix='obj_')
#     n_obj = len(obj_names)
#     obj_xyzs = sample_xyzs(
#         n_sample  = n_obj,
#         x_range   = [+0.6,+1.0],
#         y_range   = [-0.45,+0.45],
#         z_range   = [0.8,0.81],
#         min_dist  = 0.2,
#         xy_margin = 0.0
#     )
#     for obj_idx in range(n_obj):
#         env.set_p_base_body(body_name=obj_names[obj_idx],p=obj_xyzs[obj_idx,:])
#         env.set_R_base_body(body_name=obj_names[obj_idx],R=np.eye(3,3))
#     env.set_geom_color(body_names_to_color=obj_names,rgba_list=get_colors(n_obj))
#     # Loop
#     env.init_viewer(transparent=True)
#     while env.is_viewer_alive():
#         # Update
#         sliders.update()
#         env.step(ctrl=sliders.get_slider_values())
#         # Render
#         if env.loop_every(tick_every=50):
#             env.plot_time()
#             env.plot_joint_axis(axis_len=0.025,axis_r=0.005) # revolute joints
#             env.plot_links_between_bodies(rgba=(0,0,0,1),r=0.001) # link information
#             env.plot_contact_info()
#             env.render()
#     # Close slider
#     sliders.close()
#     print ("Done.")

## Inverse Kinematics of UR5e-2F85

In [11]:
if tutorial:
    sliders = MultiSliderClass( # Slider for EE control
        n_slider      = 6,
        title         = 'Sliders for [%s] Control'%(env.name),
        window_width  = 450,
        window_height = 280,
        x_offset      = 0,
        y_offset      = 100,
        slider_width  = 300,
        label_texts   = ['X','Y','Z','Roll-deg','Pitch-deg','Yaw-deg'],
        slider_mins   = [-0.4,-0.4,-0.4,-180,-180,-180],
        slider_maxs   = [+0.4,+0.4,+0.4,+180,+180,+180],
        slider_vals   = [0,0,0,0,0,0],
        resolutions   = [0.01,0.01,0.01,3.6,3.6,3.6], # range/50
        verbose       = False,
    )
    joint_names = ["shoulder_pan_joint","shoulder_lift_joint","elbow_joint","wrist_1_joint","wrist_2_joint","wrist_3_joint"]
    q0 = np.array([-1.5708,-1.5708,-1.5708,-1.5708,-1.5708,0.0])
    q0 = np.array([-0.817,-0.691,0.66,-1.63,-1.57,-3.2])
    p0 = env.get_p_body(body_name='end_effector_target')
    print("Initial EE position:[%.3f,%.3f,%.3f]"%(p0[0],p0[1],p0[2]))
    R0 = rpy_deg2r([0,0,0])
    env.init_viewer(
        title       = 'UR5e IK' ,
        transparent = False,
        azimuth     = 120,
        distance    = 1.4,
        elevation   = -24.0,
        lookat      = (0.2,0.0,0.35),
    )
    env.reset() # reset
    env.forward(q=q0,joint_names=joint_names) # initial position

    # Move object positions
    obj_names = env.get_body_names(prefix='obj_')
    n_obj = len(obj_names)
    obj_xyzs = sample_xyzs(
        n_sample  = n_obj,
        x_range   = [+0.6,+1.0],
        y_range   = [-0.45,+0.45],
        z_range   = [0.8,0.81],
        min_dist  = 0.2,
        xy_margin = 0.0
    )
    for obj_idx in range(n_obj):
        env.set_p_base_body(body_name=obj_names[obj_idx],p=obj_xyzs[obj_idx,:])
        env.set_R_base_body(body_name=obj_names[obj_idx],R=np.eye(3,3))
    env.set_geom_color(body_names_to_color=obj_names,rgba_list=get_colors(n_obj))
        
    # Loop
    q_ik_init = q0.copy()
    while env.is_viewer_alive():
        
        # Update
        sliders.update() # update slider
        xyzrpyg = sliders.get_slider_values()
        qpos,ik_err_stack,ik_info = solve_ik(
            env                = env,
            joint_names_for_ik = joint_names,
            body_name_trgt     = 'end_effector_target',
            q_init             = q_ik_init,
            p_trgt             = xyzrpyg[:3]+p0,
            R_trgt             = rpy_deg2r(xyzrpyg[3:6])@R0,
            max_ik_tick        = 500,
            ik_stepsize        = 1.0,
            ik_eps             = 1e-2,
            ik_th              = np.radians(5.0),
            render             = False,
            verbose_warning    = False,
        )
        ik_err = np.abs(ik_err_stack).max() # IK error
        if ik_err < 1e-2: q_ik_init = qpos.copy()
        else: q_ik_init = q0.copy()    
        env.forward(q=qpos,joint_names=joint_names) # kinematic update

        # Click handler
        xyz_click,flag_click = env.get_xyz_left_double_click()
        if flag_click: print ("[CLICKED] p:%s"%(xyz_click))
        
        # Render 
        if env.loop_every(tick_every=10):
            env.plot_T(
                T=env.get_T_body(body_name='world'),
                axis_len=0.5,print_xyz=False)
            env.plot_text(
                p=env.get_p_body(body_name='world')+np.array([0,0,0.5]),
                label = 'time:[%.2f]sec ik_err:[%.3f]'%(env.get_sim_time(),ik_err))
            env.plot_body_T(body_name='end_effector_target',axis_len=0.1,axis_width=0.005)
            env.plot_contact_info(
                r_arrow=0.005,h_arrow=0.1,rgba_contact=(1,0,0,0.5),plot_sphere=False)
            plot_ik_info(env=env,ik_info=ik_info)
            if xyz_click is not None:
                env.plot_sphere(p=xyz_click,r=0.01,rgba=(1,0,0,0.5))
            env.render()
        if env.loop_every(tick_every=5000): 
            img = env.grab_image()
            plt.figure(figsize=(6,4)); plt.imshow(img); 
            plt.title('tick:[%d] time:[%.2f]sec'%
                    (env.tick,env.get_sim_time()),fontsize=9)
            plt.axis('off'); plt.show()

    # Close
    env.close_viewer()
    sliders.close()
    print ("Done.")

Initial EE position:[-0.672,0.520,0.226]
env:[UR5e-2F85] initalize viewer
env:[UR5e-2F85] reset
Done.
